# Requirements

In [41]:
# Libraries
import rasterio
import geopandas as gpd

from beak.utilities.raster_processing import calculate_distance_from_raster, calculate_distance_from_raster_gdal_base_fn
from beak.utilities.io import data_folder, save_raster, check_path
from beak.utilities.conversions import create_binary_raster

# Set base data folder path
data_folder = data_folder()
vector_file_name = "SGMC_Geology_carbonate"

# Paths
cma_name = "regional_tusk_great_basin_102008_500"
base_raster_file = data_folder / "processed" / cma_name / "base_raster" / "template_raster.tif"
vector_file = data_folder / "raw" / "geology" / "sgmc" / str(vector_file_name + ".shp")
out_file = data_folder / "processed" / cma_name / "unified" / "geology" / "distance" / (vector_file_name + "_proximity.tif")

check_path(out_file.parent)


WindowsPath('S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Bearbeitung/GitHub/beak-ta3/src/beak/data/processed/regional_tusk_great_basin_102008_500/geology/distance')

In [44]:
# Load data into geodataframe
gdf = gpd.read_file(vector_file)

# Convert the geodataframe's crs to the base raster's crs
raster = rasterio.open(base_raster_file)
gdf = gdf.to_crs(raster.crs)


In [43]:
# Convert geodataframe to binary raster incl. metadata plus applying a base raster for maintaining the extent
binary_raster, meta = create_binary_raster(geodataframe=gdf,
                                           base_raster=raster,
                                           query=None,
                                           return_meta=True,
                                           )

# Calculate distance from raster
distance_array, distance_meta = calculate_distance_from_raster(input_data=(binary_raster, meta), input_mask=raster)
save_raster(out_file, distance_array, metadata=distance_meta, overwrite=True)
